# ChurnSignal — Milestone 3: Feature Engineering & Preprocessing
**Learning-First SaaS Customer Retention Intelligence**

Welcome to **Milestone 3**! Here we transform static monthly snapshots into **dynamic behavioral momentum signals**.

---

### Core ML Principle:
> *"Algorithms are blind to the passage of time unless we engineer features that capture velocity, acceleration, and recency."*

In this notebook, we implement:
1. **Delta Features (Velocity)**: $\text{Metric}_T - \text{Metric}_{T-1}$
2. **Ratio Features**: $\frac{\text{Metric}_T + 1}{\text{Metric}_{T-1} + 1}$
3. **Rolling Window Aggregations**: 3-month historical averages and maximums.
4. **Domain Risk Indicators**: Zero-seat flags, NPS detractor flags, and billing distress.
5. **Leakage-Safe Preprocessing Pipeline**: Fitting scalers and encoders exclusively on the training partition.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to sys.path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_prediction_eligible_data
from src.features.engineer import (
    compute_delta_features,
    compute_ratio_features,
    compute_rolling_features,
    compute_domain_risk_flags,
    engineer_all_features
)
from src.features.pipeline import chronological_split, ChurnFeaturePipeline

pd.set_option('display.max_columns', 35)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
print('Libraries and feature modules imported.')

## 1. Load Prediction-Eligible Data
We load the 74,817 eligible active customer-month records established in Milestone 2.


In [ ]:
raw_df = load_prediction_eligible_data()
print(f'Eligible Records: {raw_df.shape[0]:,}')
print(f'Columns: {raw_df.shape[1]}')
raw_df.head(3)

## 2. Month-over-Month Delta Features (Velocity)

### Concept Intuition:
A static metric (e.g. `sessions = 20`) tells us where the customer *is today*. 
A delta metric ($\Delta \text{sessions} = \text{sessions}_T - \text{sessions}_{T-1}$) tells us **where the customer is heading**.

Notice how customers about to churn show a usage drop nearly **3x steeper** than retained customers!


In [ ]:
df_deltas = compute_delta_features(raw_df)

print('--- Average Velocity Shifts: Retained vs Churning Next Month ---')
display(df_deltas.groupby('churned_next_month')[
    ['delta_usage', 'delta_sessions', 'delta_nps', 'delta_support_tickets', 'delta_payment_failures']
].mean().T.rename(columns={0: 'Retained Mean (0)', 1: 'Churn Mean (1)'}))

## 3. Ratio Features & Seat Utilization

### Concept Intuition:
- **Usage & Session Ratios**: Measuring proportional changes prevents large accounts from dominating linear models.
- **Seat Utilization**: $\frac{\text{sessions}}{\text{active\_seats} + 1}$ measures engagement intensity per team seat.


In [ ]:
df_ratios = compute_ratio_features(df_deltas)
display(df_ratios[['user_id', 'month', 'sessions', 'session_ratio', 'seat_utilization']].head(5))

## 4. Rolling Window Historical Aggregations

### Concept Intuition:
Single-month blips happen. A 3-month rolling window captures sustained disengagement or persistent billing friction.
- `rolling_mean_sessions_3m`: Smoothed engagement trend.
- `rolling_max_payment_failures_3m`: Flags accounts with recent billing friction even if the current month was resolved.


In [ ]:
df_rolling = compute_rolling_features(df_ratios, window=3)
display(df_rolling[['user_id', 'month', 'payment_failures', 'rolling_max_payment_failures_3m', 'rolling_mean_sessions_3m']].head(6))

## 5. Domain Risk Indicators & Collinearity Resolution

### Actions:
1. `is_zero_seats`: Paying account with zero active users.
2. `is_nps_detractor`: Net Promoter Score $\le 6$.
3. `has_payment_failure`: Billing failure in current month.
4. `is_new_customer`: Tenure 0 cold-start flag.
5. **Drop `monthly_price`**: Deduplicates the 100% collinear clone of `mrr`.


In [ ]:
df_engineered = engineer_all_features(raw_df)
print(f'Total Features After Engineering: {df_engineered.shape[1]} (from {raw_df.shape[1]} raw)')
print(f'Any missing values? {df_engineered.isnull().any().any()}')
df_engineered.head(3)

## 6. Strict Out-of-Time (OOT) Chronological Partitioning

### Validation Boundaries:
- **Train Split**: `2024-01-01` to `2025-06-01` (18 calendar months)
- **Validation Split**: `2025-07-01` to `2025-09-01` (3 calendar months)
- **Test Split**: `2025-10-01` to `2025-11-01` (2 calendar months)


In [ ]:
train_df, val_df, test_df = chronological_split(df_engineered)

print(f'Train Set      : {len(train_df):>6,} rows | Churn: {train_df["churned_next_month"].sum():>4} ({train_df["churned_next_month"].mean()*100:.2f}%)')
print(f'Validation Set : {len(val_df):>6,} rows | Churn: {val_df["churned_next_month"].sum():>4} ({val_df["churned_next_month"].mean()*100:.2f}%)')
print(f'Test Set       : {len(test_df):>6,} rows | Churn: {test_df["churned_next_month"].sum():>4} ({test_df["churned_next_month"].mean()*100:.2f}%)')
print(f'Total Accounted: {len(train_df) + len(val_df) + len(test_df):>6,} rows')

## 7. Leakage-Safe Scikit-Learn Preprocessing Pipeline

### The Golden Rule:
`ChurnFeaturePipeline` calls `.fit()` **strictly on `train_df`**, learning the mean $\mu$ and standard deviation $\sigma$ for scaling, and category encodings for `plan_type`.

It then calls `.transform()` on `val_df` and `test_df` using those frozen training parameters.


In [ ]:
pipeline = ChurnFeaturePipeline()

# Fit strictly on train!
X_train_df = pipeline.fit_transform(train_df)
X_val_df = pipeline.transform(val_df)
X_test_df = pipeline.transform(test_df)

print(f'Transformed Train Feature Shape: {X_train_df.shape}')
print(f'Transformed Val Feature Shape  : {X_val_df.shape}')
print(f'Transformed Test Feature Shape : {X_test_df.shape}')

print('\n--- Final 30 Model Input Features ---')
for idx, name in enumerate(pipeline.get_feature_names(), 1):
    print(f'{idx:>2}. {name}')

## 8. Milestone 3 Summary & Transition to Baseline Model

### Key Deliverables:
1. **Engineered 17 new features** capturing velocity, rolling history, and business risk indicators.
2. **Deduplicated `monthly_price`** to eliminate multicollinearity.
3. **Established zero-leakage chronological partitions** (Train, Validation, Test).
4. **Constructed `ChurnFeaturePipeline`** with train-only parameter fitting.

### Next Step: Milestone 4 — Baseline Modeling (Logistic Regression)
We are now ready to train our very first machine learning model!
